# CCTV 사람 속성 모델 광범위 비교

동일한 Simuletic 15장 + PA-100K 30장, 33개 그룹, 관측 라벨 분모로 실행된 유효 결과만 비교한다. 이 차트는 CCTV 동일인 식별 정확도가 아니라 사람 크롭 속성 proxy 정확도와 p95 추론시간을 보여준다. 다운로드 보류·런타임 실패·다른 metric 계약은 점수 0으로 바꾸지 않는다.

In [1]:
from pathlib import Path
import csv
import json
import math
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = Path('C:/Windows/Fonts/NotoSansKR-Regular.ttf')
if font_path.exists():
    font_manager.fontManager.addfont(str(font_path))
    plt.rcParams['font.family'] = font_manager.FontProperties(fname=str(font_path)).get_name()
plt.rcParams['axes.unicode_minus'] = False
project_root = Path.cwd()
if project_root.name == 'experiments':
    project_root = project_root.parent
result_root = project_root / 'experiments' / 'results'
parameter_billion = {'qwen3-vl-2b': 2.0, 'qwen2.5-vl-3b': 3.0, 'gemma3-4b': 4.0, 'smolvlm2-500m': 0.5, 'internvl3-2b': 2.0, 'molmo2-4b': 4.0, 'minicpmv-4.5-int4': 8.0, 'phi3.5-vision': 4.2, 'siglip2-base-224': 0.086, 'siglip2-base-384': 0.086, 'clip-vit-b32': 0.151, 'clip-vit-l14': 0.427, 'siglip-base': 0.203, 'blip-base': 0.385}
rows = []
failed = []
for path in sorted(result_root.glob('*.json')):
    result = json.loads(path.read_text(encoding='utf-8'))
    summary = result.get('summary', {})
    if result.get('status') == 'valid' and summary.get('examples') == 45 and 'attribute_accuracy' in summary:
        model = str(result.get('model', path.stem))
        rows.append({'label': model, 'x': float(summary['attribute_accuracy']) * 100, 'y': float(summary['latency_s_p95']), 'bubble': parameter_billion.get(model, 1.0), 'status': 'measured', 'source': str(path.relative_to(project_root))})
    elif result.get('status') == 'invalid_runtime':
        failed.append({'label': str(result.get('model', path.stem)), 'status': 'invalid_runtime', 'reason': str(result.get('error', ''))})
rows.sort(key=lambda row: row['x'], reverse=True)
print(f'유효 측정 모델={len(rows)}, 런타임 실패={len(failed)}')
for row in rows:
    print(f"{row['label']}: 속성={row['x']:.2f}%, p95={row['y']:.3f}s")

유효 측정 모델=7, 런타임 실패=4
clip-vit-l14: 속성=41.40%, p95=4.141s
qwen3-vl-2b: 속성=39.30%, p95=8.298s
clip-vit-b32: 속성=38.60%, p95=0.723s
siglip-base: 속성=24.21%, p95=0.778s
siglip2-base-384: 속성=21.75%, p95=8.377s
blip-base: 속성=20.35%, p95=0.468s
siglip2-base-224: 속성=20.35%, p95=4.310s


In [2]:
fig, ax = plt.subplots(figsize=(11, 6.5), constrained_layout=True)
colors = {'qwen3-vl-2b': '#d95f02', 'clip-vit-l14': '#1b9e77', 'clip-vit-b32': '#66a61e', 'siglip2-base-224': '#7570b3', 'siglip2-base-384': '#e7298a'}
for row in rows:
    ax.scatter(row['x'], row['y'], s=280 + 180 * row['bubble'], alpha=0.78, color=colors.get(row['label'], '#4c78a8'), edgecolor='black', linewidth=0.8)
    ax.annotate(row['label'], (row['x'], row['y']), xytext=(7, 7), textcoords='offset points', fontsize=9)
ax.invert_yaxis()
ax.set_title('동일 45장 CCTV 사람 크롭 proxy: 속성 정확도와 p95 지연시간')
ax.set_xlabel('관측 속성 정확도 (%)')
ax.set_ylabel('p95 추론시간 (초, 낮을수록 좋음)')
ax.grid(alpha=0.25)
chart_path = result_root / 'broad_model_comparison_proxy_bubble.png'
fig.savefig(chart_path, dpi=180)
fig.savefig(result_root / 'broad_model_comparison_proxy_bubble.svg')
plt.show()
print(f'chart={chart_path}')

<Figure size 1100x650 with 1 Axes>

chart=C:\Users\SSAFY\Desktop\2학기\qwen3vl-backend\experiments\results\broad_model_comparison_proxy_bubble.png


In [3]:
coordinate_path = result_root / 'broad_model_comparison_proxy_coordinates.csv'
with coordinate_path.open('w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=['label', 'x', 'y', 'bubble', 'status', 'source'])
    writer.writeheader()
    writer.writerows(rows)
print(f'좌표 CSV={coordinate_path}, rows={len(rows)}')
print('실패 후보')
for row in failed:
    print(f"{row['label']}: {row['reason']}")
assert all(row['status'] == 'measured' for row in rows)
assert len(rows) >= 1

좌표 CSV=C:\Users\SSAFY\Desktop\2학기\qwen3vl-backend\experiments\results\broad_model_comparison_proxy_coordinates.csv, rows=7
실패 후보
florence-2-large-ft: 'Florence2LanguageConfig' object has no attribute 'forced_bos_token_id'
florence-2-large: 'Florence2LanguageConfig' object has no attribute 'forced_bos_token_id'
gemma3-4b: Access to model google/gemma-3-4b-it is restricted and you are not in the authorized list. Visit https://huggingface.co/google/gemma-3-4b-it to ask for access.
internvl3-2b: Model type should be one of AriaConfig, AyaVisionConfig, BlipConfig, Blip2Config, ChameleonConfig, Cohere2VisionConfig, Cosmos3OmniConfig, DeepseekOcr2Config, DeepseekVLConfig, DeepseekVLHybridConfig, DiffusionGemmaConfig, Emu3Config, Ernie4_5_VLMoeConfig, EvollaConfig, Exaone4_5_Config, FastVlmConfig, Florence2Config, FuyuConfig, Gemma3Config, Gemma3nConfig, Gemma4Config, Gemma4UnifiedConfig, GitConfig, Glm46VConfig, Glm4vConfig, Glm4vMoeConfig, GlmOcrConfig, GlmgaConfig, GotOcr2Config, Granite4Vi